# Long-Short Volatility Harvest ML - Research Notebook

**Source**: QC Strategy Library - VolatilityHarvestML Long-Short

**Concept**: A long-short equity strategy combining ML regime detection (RandomForestClassifier on
11 VIX/SPY features), Hurst exponent scoring for short selection, ATR-based stops for shorts,
and a 3-stage trailing stop system for longs. The long book holds top 4 market cap stocks;
the short book targets overextended stocks with high Hurst scores (> 0.85).

In [1]:
from AlgorithmImports import *
qb = QuantBook()

# Define universe
qb.AddEquity("SPY", Resolution.Daily)
qb.AddEquity("GLD", Resolution.Daily)

print(f"Long book: Top 4 market cap stocks (monthly rebalance)")
print(f"Short book: Hurst > 0.85 + extension + momentum (weekly rebalance)")
print(f"ML model: RandomForestClassifier (11 features, VIX + SPY)")

Long book: Top 4 market cap stocks (monthly rebalance)
Short book: Hurst > 0.85 + extension + momentum (weekly rebalance)
ML model: RandomForestClassifier (11 features, VIX + SPY)


## Strategy Logic

### Long Book (daily check)
1. **Universe**: Top 4 stocks by market cap (updated monthly via fine selection)
2. **Allocation**: Equal weight from long_gross (0.9), with ML tilt (0.25) overweighting largest cap
3. **Signal**: VIX/SPY regime detection with 5 allocation states
4. **Risk**: 3-stage trailing stop (9.5% -> reduce 1/3, 7% -> reduce 2/3, 4.85% -> exit)

### Short Book (weekly rebalance)
1. **Scoring**: Hurst exponent across 6 window sizes [10, 10, 40, 60, 90, 100]
2. **Filters**: Extension (> 2x ATR above SMA) AND momentum (> 1.75x ATR above 5-day ago close)
3. **Selection**: Top N (default 1) by composite score
4. **Risk**: ATR-based stop loss (2x ATR from entry)

### ML Regime Detection
- **Model**: RandomForestClassifier (100 trees, depth 5)
- **Features**: 11 VIX/SPY metrics (level, z-score, percentile, ratios, returns, vol)
- **Label**: SPY 21-day forward return > 2% (binary classification)
- **Retraining**: Monthly on 504+ days of history
- **Usage**: When prob > 0.6, overweight longs by ml_tilt factor

In [2]:
# Fetch historical data for analysis
spy_hist = qb.History("SPY", timedelta(252 * 12), Resolution.Daily)
gld_hist = qb.History("GLD", timedelta(252 * 12), Resolution.Daily)

print(f"SPY data: {spy_hist.shape[0]} days")
print(f"GLD data: {gld_hist.shape[0]} days")

# Regime analysis: SPY vs 200-SMA
spy_close = spy_hist["close"]
spy_sma200 = spy_close.rolling(200).mean()
above_sma = (spy_close > spy_sma200).dropna()

print(f"\nSPY above 200-SMA: {above_sma.mean():.1%} of the time")
print(f"  Bull regime avg daily return: {spy_close.pct_change()[above_sma].mean():.4f}")
print(f"  Bear regime avg daily return: {spy_close.pct_change()[~above_sma].mean():.4f}")

AttributeError: 'MemoizingEnumerable[TradeBar]' object has no attribute 'shape'

In [3]:
# VIX regime analysis
import numpy as np

spy_returns = spy_close.pct_change().dropna()

print("SPY return statistics:")
print(f"  Annualized return: {(1 + spy_returns.mean())**252 - 1:.2%}")
print(f"  Annualized vol: {spy_returns.std() * (252**0.5):.2%}")
print(f"  Sharpe (rf=0): {(spy_returns.mean() / spy_returns.std()) * (252**0.5):.2f}")
print(f"  Max drawdown: {(spy_close / spy_close.cummax() - 1).min():.2%}")

NameError: name 'spy_close' is not defined

## Key Observations

- **Hurst exponent** measures mean-reversion (H < 0.5) vs trending (H > 0.5) behavior
- **ML tilt** adds conviction when RandomForest predicts bullish 21-day returns
- **3-stage trailing stops** gradually reduce exposure as a position falls from its high
- **Margin account** required for short selling (Interactive Brokers)

### Strengths
- Multi-signal approach (ML + Hurst + technical filters) reduces false positives
- Separate risk management for longs (trailing stops) and shorts (ATR stops)
- Monthly ML retraining adapts to changing volatility regimes
- VIX-based allocation shifts between equities and gold dynamically
- Long book uses large-cap stocks (low idiosyncratic risk)

### Risks
- Complex strategy with many parameters (overfitting risk)
- Margin account amplifies losses on both long and short sides
- Hurst exponent estimation is noisy over short windows
- Short book concentrated (default top_n=1) has high stock-specific risk
- CBOE VIX data source may have reliability/lag issues in live trading
- 11-feature ML model with 504-day training may not generalize across regimes

In [4]:
# Backtest results placeholder
print("=" * 50)
print("BACKTEST RESULTS (12-year)")
print("=" * 50)
print("Run backtest via QC MCP to populate metrics:")
print("  create_compile -> create_backtest -> read_backtest")
print()
print("Expected characteristics:")
print("  - CAGR: ~12-20% (long-short with leverage)")
print("  - Max Drawdown: ~15-25%")
print("  - Sharpe: ~0.8-1.3")
print("  - Rebalance: Daily signal, weekly shorts, monthly ML retrain")
print("  - Universe: 2000 coarse -> 150 fine -> 4 long + 1 short")
print("  - Margin account required for short selling")

BACKTEST RESULTS (12-year)
Run backtest via QC MCP to populate metrics:
  create_compile -> create_backtest -> read_backtest

Expected characteristics:
  - CAGR: ~12-20% (long-short with leverage)
  - Max Drawdown: ~15-25%
  - Sharpe: ~0.8-1.3
  - Rebalance: Daily signal, weekly shorts, monthly ML retrain
  - Universe: 2000 coarse -> 150 fine -> 4 long + 1 short
  - Margin account required for short selling
